# 01 · Download Kerala buildings and district boundaries

**Run this in Google Colab, not on your laptop.**

What it does:
1. Connects your Google Drive. Files are saved in `MyDrive/rooftop_solar/data/raw`.
2. Downloads Kerala's state boundary and its 14 district boundaries from Overture Maps.
3. Downloads every building footprint in a box around Kerala from Overture Maps (release 2026-10-21.0).
4. Prints a short summary.

**How to run:** Runtime → Run all. Allow Google Drive access when asked. It takes about 10–30 minutes.

When it finishes, copy the tables printed in Step 3 and Step 5 and send them to Claude.

In [ ]:
# Step 1: connect Google Drive and make the project folders
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
RAW = f'{PROJECT}/data/raw'   # permanent copy on Google Drive
LOCAL = '/content/raw'        # fast Colab disk, deleted when the session ends
os.makedirs(RAW, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)
print('Project folder:', PROJECT)

In [ ]:
# Step 2: install DuckDB (reads Overture files straight from the cloud, no login needed)
!pip install -q duckdb==2.0.0

import duckdb

con = duckdb.connect()
con.sql("INSTALL spatial; LOAD spatial; INSTALL httpfs; LOAD httpfs;")
con.sql("SET s3_region='us-west-2'; SET preserve_insertion_order=false;")


def run(query):
    """Run a SQL statement and wait until it finishes."""
    return con.execute(query).fetchall()


RELEASE = '2026-10-21.0'
BASE = f's3://overturemaps-us-west-2/release/{RELEASE}'

# Box around Kerala. It also catches some Tamil Nadu and Karnataka buildings;
# notebook 02 removes them with a Sedona spatial join on the district boundaries.
XMIN, XMAX, YMIN, YMAX = 74.85, 77.45, 8.15, 12.80
print('DuckDB', duckdb.__version__)

In [ ]:
# Step 3: Kerala state boundary (subtype 'region') + 14 districts (subtype 'county')
t0 = time.time()
run(f"""
    COPY (
        SELECT id, names.primary AS name, subtype, geometry
        FROM read_parquet('{BASE}/theme=divisions/type=division_area/*', hive_partitioning=1)
        WHERE country = 'IN' AND region = 'IN-KL' AND is_land
          AND subtype IN ('region', 'county')
          AND bbox.xmin < {XMAX} AND bbox.xmax > {XMIN}
          AND bbox.ymin < {YMAX} AND bbox.ymax > {YMIN}
    ) TO '{LOCAL}/kerala_boundaries.parquet' (FORMAT parquet)
""")
print(con.sql(f"""
    SELECT subtype, count(*) AS n, string_agg(name, ', ' ORDER BY name) AS names
    FROM '{LOCAL}/kerala_boundaries.parquet'
    GROUP BY subtype
"""))
print(f'Done in {time.time() - t0:.0f} s  (expected: county = 14, region = 1)')

In [ ]:
# Step 4: every building footprint in the Kerala box (the big download)
t0 = time.time()
run(f"""
    COPY (
        SELECT id, subtype, class, height, num_floors,
               roof_shape, roof_material, roof_direction, roof_orientation,
               sources[1].dataset AS source,
               sources[1].confidence AS confidence,
               bbox, geometry
        FROM read_parquet('{BASE}/theme=buildings/type=building/*', hive_partitioning=1)
        WHERE bbox.xmin BETWEEN {XMIN} AND {XMAX}
          AND bbox.ymin BETWEEN {YMIN} AND {YMAX}
    ) TO '{LOCAL}/buildings_kerala_box.parquet' (FORMAT parquet, COMPRESSION zstd)
""")
size_gb = os.path.getsize(f'{LOCAL}/buildings_kerala_box.parquet') / 1e9
print(f'Done in {(time.time() - t0) / 60:.1f} min, file size {size_gb:.2f} GB')

In [ ]:
# Step 5: summary to send to Claude
# Note: these counts cover the whole box, so they include some Tamil Nadu and Karnataka buildings.
B = f"'{LOCAL}/buildings_kerala_box.parquet'"
print(con.sql(f"""
    SELECT count(*)                                   AS buildings,
           count(height)                              AS with_height,
           round(100.0 * count(height) / count(*), 1) AS pct_height,
           count(num_floors)                          AS with_floors,
           count(class)                               AS with_class,
           count(roof_shape)                          AS with_roof_shape
    FROM {B}
"""))
print(con.sql(f"SELECT source, count(*) AS n FROM {B} GROUP BY source ORDER BY n DESC"))
# How sure the AI was that each shape is a building (Google: 0.65 to 1.0)
print(con.sql(f"""
    SELECT source, floor(confidence * 20) / 20 AS confidence_band, count(*) AS n
    FROM {B} WHERE confidence IS NOT NULL
    GROUP BY ALL ORDER BY source, confidence_band
"""))
print(con.sql(f"""
    SELECT class, count(*) AS n FROM {B}
    WHERE class IS NOT NULL
    GROUP BY class ORDER BY n DESC LIMIT 10
"""))

In [ ]:
# Step 6: copy the files to Google Drive so they survive after Colab closes
for name in os.listdir(LOCAL):
    shutil.copy(f'{LOCAL}/{name}', f'{RAW}/{name}')
    print('Saved:', f'{RAW}/{name}')

## Done

Send Claude the tables from **Step 3** and **Step 5**, or the error message if a step failed.

The exact number of buildings inside Kerala comes in notebook 02, where Sedona joins the buildings to the district boundaries.